<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Wan2.2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Wan 2.2 — Text & Image-to-Video (5B + 14B-A14B)

A self-contained Colab wrapper around [Tencent / Wan-AI's Wan 2.2](https://github.com/Wan-Video/Wan2.2) video model. Pick from two officially-released variants, then generate 720p @ 24 fps videos from a text prompt, an input image, or both.

- **Two official variants in one notebook**
  - **5B dense** (default, 10 GB on disk, fits 24 GB GPUs with sequential CPU offload) — fast, recommended for most Colab users
  - **14B-A14B MoE** (heavy, 27 GB on disk, needs 40+ GB VRAM) — highest quality, slow
- **Two modes per variant**: Text-to-Video (T2V) and Image-to-Video (I2V)
- **720p @ 24 fps output**, up to 5 seconds (121 frames) per call
- **Apache 2.0 license** on both variants
- **Official diffusers port** — uses `WanPipeline` + `AutoencoderKLWan` from the `diffusers` main branch
- **Drive-cached weights** — 10 GB (5B) or 27 GB (14B) on first run, then instant

### Enhancement features (added in this notebook)

- **6 preset style buttons** (Cinematic / Animation / Nature / Slow Motion / Action / Portrait) that prepend a tuned style prefix to your prompt with one click
- **Resolution preset dropdown** — pick from 6 common aspect ratios or drop to Custom for the sliders
- **Negative prompt field** — most video models benefit from one. Default is tuned to the upstream Space
- **Auto-enhance prompt toggle** — when on, prepends a `smooth, fluid motion` baseline to plain prompts (off by default)
- **Smart resolution auto-pick** — when you upload an image, the dropdown auto-snaps to the closest matching preset

### Quick start

1. Pick the **L4 (24 GB)** runtime for the 5B variant, or **A100 (40 GB)** for the 14B-A14B. T4 (16 GB) can run the 5B at 480p with offload but it will be slow.
2. Run Steps 1-4 in order — Step 1 installs, Step 2 caches the chosen variant's weights to Drive, Step 3 defines the pipeline, Step 4 launches the Gradio UI.
3. Open the public Gradio URL, type a prompt (and optionally upload an image), pick a preset style if you want, and click **Generate**.

**Runtime:** a 2-second 720p clip takes 3-8 minutes on an L4 (5B), or 10-30 minutes on an A100 (14B).

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs `diffusers` from the main branch (Wan 2.2 needs features not yet on PyPI),
# @markdown plus `transformers`, `accelerate`, and `huggingface_hub` for the cache.

import os, sys, subprocess, time, importlib

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime \u2192 Change runtime type \u2192 L4 or A100')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} \u2014 {vram_gb:.0f} GB')

print('[pip] Installing diffusers from main branch (Wan 2.2 features not yet on PyPI) ...')
t0 = time.time()
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/huggingface/diffusers',
    'transformers>=4.48.0',
    'accelerate>=1.0.0',
    'huggingface-hub>=0.25.0',
    'imageio[ffmpeg]',
    'av',
    'sentencepiece',
    'tqdm',
    'gradio==5.33.0',
], check=True)
print(f'[Install] DONE in {time.time() - t0:.1f}s.')

try:
    from diffusers import WanPipeline, AutoencoderKLWan, WanTransformer3DModel
    from diffusers.utils import export_to_video, load_image
    print('[Install] diffusers Wan classes OK.')
except Exception as e:
    raise SystemExit(
        'diffusers main-branch install did not expose Wan classes.\n'
        'Try running Step 1 again, or `pip install --upgrade git+https://github.com/huggingface/diffusers`\n'
        f'Error: {e}'
    )

In [ ]:
# @title STEP 2 — Pre-cache Weights
# @markdown Downloads the chosen variant's weights to Google Drive so subsequent runs are instant.
# @markdown Also mounts Drive for output caching.

import os, sys, pathlib, time, shutil
from huggingface_hub import snapshot_download
from tqdm.notebook import tqdm


# --- Model registry ---
# Each entry: (hf_repo, on_disk_gb, recommended_vram_gb, blurb)
MODEL_REGISTRY = {
    '5B-TI2V':  ('Wan-AI/Wan2.2-TI2V-5B-Diffusers',  10, 24, '5B dense, 720p@24fps, fits 24GB GPUs with offload'),
    '14B-A14B': ('Wan-AI/Wan2.2-I2V-A14B-Diffusers', 27, 40, '14B MoE (2x14B experts, 14B active), highest quality, needs 40+ GB VRAM'),
}
DEFAULT_VARIANT = '5B-TI2V'


REPO, ON_DISK_GB, RECOMMENDED_VRAM_GB, BLURB = MODEL_REGISTRY[DEFAULT_VARIANT]
print(f'[Cache] Default variant: {DEFAULT_VARIANT} ({REPO})')
print(f'[Cache] \u2248 {ON_DISK_GB} GB on disk, recommends \u2265 {RECOMMENDED_VRAM_GB} GB VRAM.')
print(f'[Cache] {BLURB}')

try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache/huggingface/hub')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(CACHE_ROOT.parent)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT)
    print(f'[Cache] HF cache root: {CACHE_ROOT}')
    HAS_DRIVE = True
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface (in-session only).')
    HAS_DRIVE = False

OUT_DIR = pathlib.Path('/content/wan22_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if HAS_DRIVE:
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan2.2')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'[Drive] Video cache: {DRIVE_OUT}')
else:
    DRIVE_OUT = OUT_DIR

def _human(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def _dir_size(p):
    p = pathlib.Path(p)
    if not p.exists(): return 0
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

def download_variant(variant_key, force=False):
    repo, on_disk_gb, _, _ = MODEL_REGISTRY[variant_key]
    local_dir = pathlib.Path(CACHE_ROOT) / repo.replace('/', '_')
    existing = _dir_size(local_dir)
    # For the 14B we need ~25 GB cached to skip the download
    threshold = max(8_000_000_000, on_disk_gb * 1_000_000_000 * 0.8)
    if existing > threshold and not force:
        print(f'[Cache] {variant_key} already cached ({_human(existing)}). Skipping download.')
        return str(local_dir)
    print(f'[Cache] Downloading {variant_key} from {repo} (~{on_disk_gb} GB) ...')
    t0 = time.time()
    snapshot_download(
        repo_id=repo,
        local_dir=str(local_dir),
        local_dir_use_symlinks=False,
        tqdm_class=tqdm,
    )
    print(f'[Cache] DONE in {time.time() - t0:.1f}s. Local: {local_dir}')
    return str(local_dir)

# Download the default variant. The 14B is on demand \u2014 the user picks it in the UI before clicking Generate.
ckpt_dir = download_variant(DEFAULT_VARIANT)
print(f'\n[Setup] Default weights at: {ckpt_dir}')
print(f'[Setup] To use the 14B variant, pick it in the UI and click Generate \u2014 it will download on demand.')
print(f'[Setup] Output videos will go to: {DRIVE_OUT}')
print('Run Step 3 next.')

In [ ]:
# @title STEP 3 — Pipeline (Lazy Model Loading + Variant Switching)
# @markdown Defines a thin `Wan22Engine` class that holds the diffusers pipeline and can switch between
# @markdown the 5B and 14B-A14B variants on demand. CPU offload is used to fit 24 GB GPUs.

import os, sys, time, gc, traceback, json, uuid, pathlib, random, tempfile
import torch, numpy as np
from PIL import Image
from IPython.display import display, FileLink
from diffusers import WanPipeline, AutoencoderKLWan, UniPCMultistepScheduler
from diffusers.utils import export_to_video, load_image

DEFAULT_FPS = 24
MOD_VALUE = 32  # height/width must be multiples of 32
NEGATIVE_PROMPT_DEFAULT = ('色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，'

                         '整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，'

                         '画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，'

                         '静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走')
AUTO_ENHANCE_PREFIX = 'Generate a video with smooth and natural movement. Objects should have visible motion while maintaining fluid transitions.'

class Wan22Engine:
    """Lazy loader for the Wan 2.2 TI2V / I2V-A14B pipeline. Variant-switchable."""
    def __init__(self):
        self.pipeline = None
        self.vae = None
        self.variant = None  # key from MODEL_REGISTRY
        self.dtype = torch.bfloat16

    def load(self, variant='5B-TI2V', offload=True, force_reload=False):
        """Load (or switch to) the requested variant."""
        if variant not in MODEL_REGISTRY:
            raise ValueError(f'unknown variant: {variant}; choose from {list(MODEL_REGISTRY)}')
        if self.pipeline is not None and self.variant == variant and not force_reload:
            return
        if self.pipeline is not None:
            self.unload()
        repo, on_disk_gb, rec_vram, _ = MODEL_REGISTRY[variant]
        if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory / 1024**3 < rec_vram * 0.7:
            print(f'[Load] WARNING: this variant recommends \u2265 {rec_vram} GB VRAM; you have less. Expect OOMs.')
        # Ensure the weights are cached locally before constructing the pipeline.
        local_dir = pathlib.Path(CACHE_ROOT) / repo.replace('/', '_')
        if not local_dir.exists() or not any(local_dir.iterdir()):
            print(f'[Load] {variant} weights not cached yet; downloading ~{on_disk_gb} GB ...')
            download_variant(variant)
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'[Load] {variant} from {local_dir} (offload={offload}) ...')
        t0 = time.time()
        try:
            # The high-compression Wan2.2 VAE must stay in fp32 for numerical stability.
            self.vae = AutoencoderKLWan.from_pretrained(
                str(local_dir), subfolder='vae', torch_dtype=torch.float32)
            self.pipeline = WanPipeline.from_pretrained(
                str(local_dir), vae=self.vae, torch_dtype=self.dtype)
            self.pipeline.scheduler = UniPCMultistepScheduler.from_config(
                self.pipeline.scheduler.config)
            if offload:
                self.pipeline.enable_sequential_cpu_offload()
            else:
                self.pipeline.to('cuda')
            self.variant = variant
        except Exception as e:
            print(f'[Load] FAILED: {e}'); raise
        print(f'[Load] Ready in {time.time() - t0:.1f}s. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

    def unload(self):
        if self.pipeline is not None:
            del self.pipeline; self.pipeline = None
        if self.vae is not None:
            del self.vae; self.vae = None
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        self.variant = None
        print('[Engine] Unloaded. VRAM: ' + f"{torch.cuda.memory_allocated()/1024**3:.1f} GB")

    def generate(self,
                prompt: str,
                negative_prompt: str = '',
                image = None,
                height: int = 704,
                width: int = 1280,
                num_frames: int = 121,
                num_inference_steps: int = 38,
                guidance_scale: float = 5.0,
                shift: float = 5.0,
                seed: int = 42,
                on_progress=None) -> str:
        """Run inference. Returns the path to the .mp4 output."""
        if self.pipeline is None:
            raise RuntimeError('engine not loaded; pick a variant in the UI first')
        g = torch.Generator().manual_seed(int(seed))
        kwargs = dict(
            prompt=prompt,
            negative_prompt=negative_prompt or None,
            height=int(height),
            width=int(width),
            num_frames=int(num_frames),
            guidance_scale=float(guidance_scale),
            num_inference_steps=int(num_inference_steps),
            generator=g,
        )
        if image is not None:
            kwargs['image'] = image
        if on_progress: on_progress(0.05, 'Running denoising ...')
        out = self.pipeline(**kwargs)
        if on_progress: on_progress(0.95, 'Encoding .mp4 ...')
        out_path = OUT_DIR / f'{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4'
        export_to_video(out.frames[0], str(out_path), fps=DEFAULT_FPS)
        return str(out_path)

ENGINE = Wan22Engine()
print('[Core] Ready. Engine loads on first Generate click.')

In [ ]:

# @title STEP 4 — Launch Gradio UI
# @markdown Opens a Gradio app with T2V and I2V tabs, model-variant selector, and the enhanced prompt tools.

import os, sys, time, json, uuid, traceback, pathlib, random, re
import torch, numpy as np, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink

# ---------- Resolution presets (H, W) ----------
RES_PRESETS = [
    ('1280 × 704 (landscape, 720p)',   704, 1280),
    ('704 × 1280 (portrait, 720p)',   1280, 704),
    ('1280 × 720 (16:9, 720p)',       720, 1280),
    ('720 × 1280 (9:16, 720p)',       1280, 720),
    ('832 × 480 (landscape, 480p)',   480, 832),
    ('480 × 832 (portrait, 480p)',     832, 480),
    ('Custom (use the sliders)',            0,   0),
]
RES_PRESET_NAMES = [p[0] for p in RES_PRESETS]
DEFAULT_RES_NAME = RES_PRESET_NAMES[0]

# ---------- Preset style buttons (click to apply) ----------
STYLE_TEMPLATES = {
    'Cinematic':   'cinematic shot of {subject}, professional lighting, smooth camera movement, 4k quality',
    'Animation':   'animated style {subject}, vibrant colors, fluid motion, dynamic movement',
    'Nature':      'nature documentary footage of {subject}, wildlife photography, natural movement',
    'Slow Motion': 'slow motion capture of {subject}, high speed camera, detailed motion',
    'Action':      'dynamic action shot of {subject}, fast paced movement, energetic motion',
    'Portrait':    'portrait of {subject}, shallow depth of field, soft natural light, gentle motion',
}


def _snap_dim(v, mod=MOD_VALUE, lo=128, hi=1280):
    v = max(lo, min(hi, int(v)))
    v = max(mod, (v // mod) * mod)
    return v


def _new_out() -> pathlib.Path:
    p = DRIVE_OUT / f'wan22_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4'
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def _err(msg: str):
    return None, 'ERROR: ' + msg


def _calc_dims_from_image(img, calc_max_area=1280.0 * 704.0):
    """Auto-pick output H/W preserving the input image's aspect ratio."""
    if img is None:
        return DEFAULT_RES_PRESETS_H, DEFAULT_RES_PRESETS_W
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img).convert('RGB')
    orig_w, orig_h = img.size
    if orig_w <= 0 or orig_h <= 0:
        return DEFAULT_RES_PRESETS_H, DEFAULT_RES_PRESETS_W
    aspect = orig_h / orig_w
    calc_h = round(np.sqrt(calc_max_area * aspect))
    calc_w = round(np.sqrt(calc_max_area / aspect))
    return _snap_dim(calc_h), _snap_dim(calc_w)


def _apply_template(template, current_prompt):
    """Append the template to the current prompt, or substitute {subject} if present."""
    subject = (current_prompt or '').split(',')[0].strip() or 'a subject of your choice'
    if '{subject}' in template:
        return template.replace('{subject}', subject)
    return (template + ' ' + current_prompt).strip()


def _maybe_auto_enhance(prompt, enable):
    """Prepend the standard motion-emphasizing prefix if enabled and the user prompt is plain."""
    if not enable:
        return prompt
    p = (prompt or '').strip()
    # If the prompt already has motion / camera keywords, don't double up.
    motion_words = ('motion', 'move', 'moving', 'camera', 'pan', 'zoom', 'fly', 'walk', 'run', 'drift', 'flow')
    if any(w in p.lower() for w in motion_words):
        return p
    return AUTO_ENHANCE_PREFIX + ' ' + p


def _resolve_resolution(res_name, h_in, w_in):
    """Given the dropdown selection (or 'Custom'), return the (h, w) to use.
    If 'Custom', fall through to the slider values."""
    for name, h, w in RES_PRESETS:
        if name == res_name:
            if h == 0:  # Custom
                return _snap_dim(h_in), _snap_dim(w_in)
            return h, w
    return _snap_dim(h_in), _snap_dim(w_in)


def on_image_upload_dims(img, current_res_name):
    """When an image is uploaded, auto-pick the preset closest to its aspect ratio."""
    if img is None:
        return gr.update(value=DEFAULT_RES_NAME)
    h, w = _calc_dims_from_image(img)
    # Find the matching preset (height, width) in RES_PRESETS
    for name, ph, pw in RES_PRESETS:
        if ph == h and pw == w:
            return gr.update(value=name)
    return gr.update(value='Custom (use the sliders)')


def on_res_preset_change(res_name):
    """When the user picks a preset, update the sliders (but keep the dropdown visible)."""
    for name, h, w in RES_PRESETS:
        if name == res_name and h != 0:
            return gr.update(value=h), gr.update(value=w)
    return gr.update(), gr.update()


def do_generate(variant, prompt, negative_prompt, auto_enhance, res_name, height, width,
                duration, steps, guidance, shift, seed, offload, image,
                progress=gr.Progress()):
    if not prompt or not prompt.strip():
        return _err('Type a prompt first.')
    try:
        # 1) Load (or switch to) the chosen variant. This may download the 14B on first use.
        progress(0.02, desc='Loading ' + variant + ' (first run may download the weights) ...')
        ENGINE.load(variant=variant, offload=offload)
        # 2) Optionally prepend the motion-emphasizing prefix.
        full_prompt = _maybe_auto_enhance(prompt, auto_enhance)
        if auto_enhance and full_prompt != prompt:
            progress(0.05, desc='Auto-enhance prepend: "' + AUTO_ENHANCE_PREFIX[:40] + '...\\u2026"')
        # 3) Resolve resolution from the preset.
        h, w = _resolve_resolution(res_name, height, width)
        # 4) Resize the input image (if I2V) to the chosen output dims.
        input_img = None
        if image is not None:
            if not isinstance(image, Image.Image):
                input_img = Image.fromarray(image).convert('RGB')
            else:
                input_img = image.convert('RGB')
            input_img = input_img.resize((w, h))
        # 5) Run.
        progress(0.10, desc='Denoising ' + str(int(duration) * 24) + ' frames at ' + str(h) + 'x' + str(w) + ' ...')
        t0 = time.time()
        out_path = ENGINE.generate(
            prompt=full_prompt,
            negative_prompt=negative_prompt or NEGATIVE_PROMPT_DEFAULT,
            image=input_img,
            height=h,
            width=w,
            num_frames=int(round(float(duration) * 24)),
            num_inference_steps=int(steps),
            guidance_scale=float(guidance),
            shift=float(shift),
            seed=int(seed),
        )
        final = _new_out()
        shutil.copy2(out_path, str(final))
        progress(1.0, desc='Done.')
        return str(final), 'Generated in ' + str(round(time.time() - t0, 1)) + 's (' + variant + ') \\u2192 ' + str(final)
    except Exception as e:
        traceback.print_exc()
        return _err(str(e))


def vram_status():
    if not torch.cuda.is_available():
        return 'No GPU.'
    a = torch.cuda.memory_allocated() / 1024**3
    r = torch.cuda.memory_reserved() / 1024**3
    t = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return (f'Allocated: {a:.1f} GB\\nReserved:  {r:.1f} GB\\nTotal:     {t:.1f} GB\\n'
            f'Pipeline loaded: {ENGINE.variant or chr(8212)}')


def free_engine():
    ENGINE.unload()
    return f'Freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.'


EXAMPLE_PROMPTS = [
    'Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage.',
    'A cinematic shot of a boat sailing on a calm sea at sunset, golden hour light.',
    'Drone footage flying over a futuristic city with flying cars, neon lights.',
    'A white cat wearing sunglasses sits on a surfboard at a sunny beach, relaxed mood.',
    'A red rose blooming in slow motion, macro lens, soft studio lighting.',
    'Astronaut riding a horse on the moon, Earth in the background, cinematic.',
]


# Pre-resolve the default 720p landscape dims.
DEFAULT_RES_PRESETS_H, DEFAULT_RES_PRESETS_W = 704, 1280


with gr.Blocks(title='Wan 2.2 Colab', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Wan 2.2 \\u2014 Text & Image-to-Video\\nTwo variants. Two modes. Apache 2.0.')

    gr.HTML('<div style="background: #fef3c7; color: #92400e; padding: 8px 12px; border-radius: 6px; margin: 6px 0;">'
            '<b>Pick a variant:</b> <b>5B</b> fits 24 GB GPUs (L4 / RTX 4090) with offload. '
            '<b>14B-A14B</b> is the highest-quality MoE and needs 40+ GB VRAM (A100) and a 27 GB download. '
            'On T4 (16 GB) use 5B at 480p only.'
            '</div>')

    with gr.Tabs():
        # -- Main tab (shared T2V + I2V; image is optional) --
        with gr.Tab('Generate'):
            with gr.Row():
                with gr.Column(scale=2):
                    variant_dd = gr.Dropdown(
                        choices=list(MODEL_REGISTRY.keys()),
                        value=DEFAULT_VARIANT,
                        label='Model variant',
                        info='5B = fast, 24 GB GPUs. 14B-A14B = highest quality, 40+ GB GPUs. 14B downloads 27 GB on first use.'
                    )
                    prompt_in = gr.Textbox(
                        label='Prompt',
                        value='Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage.',
                        lines=4,
                        info='Describe the scene in detail: subjects, action, setting, lighting, camera angle. The model follows detailed prompts well.'
                    )
                    # Preset style chips
                    with gr.Accordion('Preset styles (click to apply)', open=False):
                        gr.Markdown('Each button prepends a tuned style to your subject. Your text becomes the `{subject}`.')
                        style_buttons = {}
                        for name in STYLE_TEMPLATES:
                            btn = gr.Button(name, size='sm')
                            style_buttons[name] = btn
                        for name, btn in style_buttons.items():
                            btn.click(
                                fn=lambda n=name, t=STYLE_TEMPLATES[name], p=prompt_in: _apply_template(t, p),
                                inputs=[prompt_in],
                                outputs=[prompt_in],
                            )
                    auto_enhance = gr.Checkbox(
                        value=False,
                        label='Auto-enhance plain prompts',
                        info='When on, prepends a `smooth natural movement` baseline if your prompt has no motion keywords.'
                    )
                    image_in = gr.Image(label='Input image (optional \\u2014 leave blank for T2V)', type='pil', image_mode='RGB', height=280)
                    image_in.upload(on_image_upload_dims, inputs=[image_in, res_dd], outputs=[res_dd])
                    image_in.clear(on_image_upload_dims, inputs=[image_in, res_dd], outputs=[res_dd])
                    with gr.Row():
                        res_dd = gr.Dropdown(
                            choices=RES_PRESET_NAMES,
                            value=DEFAULT_RES_NAME,
                            label='Resolution',
                            info='Both dimensions are multiples of 32. Auto-snaps to closest preset when you upload an image.'
                        )
                    with gr.Row():
                        height_in = gr.Slider(128, 1280, value=DEFAULT_RES_PRESETS_H, step=MOD_VALUE, label='Height (used when Resolution = Custom)')
                        width_in = gr.Slider(128, 1280, value=DEFAULT_RES_PRESETS_W, step=MOD_VALUE, label='Width (used when Resolution = Custom)')
                    res_dd.change(on_res_preset_change, inputs=[res_dd], outputs=[height_in, width_in])
                    duration_in = gr.Slider(0.3, 5.0, value=2.0, step=0.1, label='Duration (seconds)',
                                            info='At 24 fps, 2.0s = 48 frames, 5.0s = 121 frames (max). Longer = much slower.')
                    with gr.Accordion('Advanced options', open=False):
                        neg_prompt = gr.Textbox(label='Negative prompt', value=NEGATIVE_PROMPT_DEFAULT, lines=3,
                                                  info='Things to avoid. Default is tuned to the upstream Space. Set empty to use diffusers default.')
                        steps_in = gr.Slider(10, 50, value=38, step=1, label='Inference steps',
                                              info='Default 38 is a good balance. Lower (20-25) for speed, higher (45-50) for quality.')
                        guidance_in = gr.Slider(1.0, 10.0, value=5.0, step=0.1, label='Guidance scale',
                                                info='Higher = stronger prompt adherence but more artifacts. 5.0 is a good default.')
                        shift_in = gr.Slider(1.0, 20.0, value=5.0, step=0.1, label='Sample shift',
                                            info='Higher = more dynamic motion. 5.0 is a good default for Wan 2.2.')
                        seed_in = gr.Slider(-1, 2_147_483_647, value=-1, step=1, label='Seed',
                                            info='-1 for random. Any fixed value for reproducibility.')
                        offload_in = gr.Checkbox(value=True, label='Enable sequential CPU offload',
                                                    info='Required for 24 GB GPUs. Set False only if you have 40+ GB VRAM (e.g. A100 with 14B).')
                    btn = gr.Button('Generate Video', variant='primary')
                with gr.Column(scale=3):
                    video_out = gr.Video(label='Generated video', height=520)
                    status_out = gr.Markdown('')
            btn.click(do_generate,
                      inputs=[variant_dd, prompt_in, neg_prompt, auto_enhance, res_dd, height_in, width_in,
                              duration_in, steps_in, guidance_in, shift_in, seed_in, offload_in, image_in],
                      outputs=[video_out, status_out])

        # -- VRAM / status tab --
        with gr.Tab('VRAM'):
            gr.Markdown('Use these buttons to free the pipeline from VRAM. The 5B uses ~10 GB of weights, the 14B uses ~27 GB.')
            vram_text = gr.Textbox(label='Status', value=vram_status, interactive=False, lines=5)
            with gr.Row():
                gr.Button('Free pipeline').click(free_engine, outputs=vram_text)
                gr.Button('Refresh status').click(vram_status, outputs=vram_text)

    gr.Examples(label='Quick starts', examples=[[p] for p in EXAMPLE_PROMPTS], inputs=[prompt_in], examples_per_page=6)

clear_output()
print('[UI] Launching ...')
demo.queue(max_size=4, default_concurrency_limit=2)
demo.load(lambda: 'Wan 2.2 is ready. Pick a variant, type a prompt, click Generate. Pipeline loads on first click (5-10s).', outputs=[status_out])
demo.launch(share=True, show_error=True, height=820, prevent_thread_lock=False)
print('\\n[UI] Public URL above. Open it in a new tab.')

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings a tiny URL every 60 s to keep the Colab tab active.

import time, requests, datetime, threading, IPython
_stop = threading.Event()
def _pinger():
    while not _stop.is_set():
        try:
            requests.get('https://huggingface.co/api/models/Wan-AI/Wan2.2-TI2V-5B-Diffusers', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} OK. Stop with `_stop.set()`.')
        except Exception as e:
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} WARN: {e}')
        _stop.wait(60)
_t = threading.Thread(target=_pinger, daemon=True)
_t.start()
print('[Keep-Alive] Started.')

In [ ]:
# @title STEP 6 — Quick Test (single T2V inference, no UI)
# @markdown Runs a single end-to-end video generation. No UI required.

import os, time, traceback, pathlib
from IPython.display import display, FileLink

QUICK_VARIANT = '5B-TI2V'  # @param ['5B-TI2V', '14B-A14B']
QUICK_PROMPT = 'A cinematic shot of a boat sailing on a calm sea at sunset, golden hour light.'  # @param {type:\"string\"}
QUICK_HEIGHT = 480  # @param {type:\"integer\"}
QUICK_WIDTH = 832  # @param {type:\"integer\"}
QUICK_DURATION = 1.5  # @param {type:\"number\"}
QUICK_STEPS = 25  # @param {type:\"integer\"}
QUICK_SEED = 42  # @param {type:\"integer\"}

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected.')

t0 = time.time()
try:
    print(f'[QuickTest] Loading {QUICK_VARIANT} ...')
    ENGINE.load(variant=QUICK_VARIANT, offload=True)
    print(f'[QuickTest] Generating {QUICK_DURATION}s clip at {QUICK_HEIGHT}x{QUICK_WIDTH} ...')
    out_path = ENGINE.generate(
        prompt=QUICK_PROMPT,
        image=None,
        height=QUICK_HEIGHT,
        width=QUICK_WIDTH,
        num_frames=int(round(QUICK_DURATION * 24)),
        num_inference_steps=QUICK_STEPS,
        seed=QUICK_SEED,
    )
    final = DRIVE_OUT / f'quicktest_{int(time.time())}.mp4'
    shutil.copy2(out_path, str(final))
    print(f'[QuickTest] DONE in {time.time() - t0:.1f}s. {final}')
    display(FileLink(str(final)))
except Exception as e:
    print(f'[QuickTest] FAILED: {e}'); traceback.print_exc()
    raise

In [ ]:

# @title STEP 7 — Batch Generation from a .txt of prompts
# @markdown Reads a text file (one prompt per line) and generates a .mp4 for each.
# @markdown Per-item try/except so one failure doesn't stop the batch. Outputs go to DRIVE_OUT.

import os, time, traceback, pathlib

# --- Edit these before running ---------------------------------
BATCH_VARIANT = '5B-TI2V'  # @param ['5B-TI2V', '14B-A14B']
BATCH_INPUT = '/content/wan22_batch_prompts.txt'  # one prompt per line
BATCH_RES_NAME = '1280 × 704 (landscape, 720p)'  # resolution preset or 'Custom (use the sliders)'
BATCH_HEIGHT = 480  # used if BATCH_RES_NAME = Custom
BATCH_WIDTH = 832
BATCH_DURATION = 1.5
BATCH_STEPS = 25
BATCH_SEED = 42
BATCH_OFFLOAD = True
MAX_ITEMS = 0  # 0 = no cap
# --------------------------------------------------------------

list_path = pathlib.Path(BATCH_INPUT)
if not list_path.exists():
    examples = [
        'A cinematic shot of a boat sailing on a calm sea at sunset, golden hour light.',
        'Drone footage flying over a futuristic city with flying cars, neon lights.',
        'A white cat wearing sunglasses sits on a surfboard at a sunny beach.',
        'A red rose blooming in slow motion, macro lens, soft studio lighting.',
        'Astronaut riding a horse on the moon, Earth in the background.',
    ]
    list_path.parent.mkdir(parents=True, exist_ok=True)
    list_path.write_text('\n'.join(examples) + '\n')
    print(f'[Batch] No list file found, bootstrapped {len(examples)} examples into {list_path}')

items = [l.strip() for l in list_path.read_text().splitlines() if l.strip() and not l.startswith('#')]
if MAX_ITEMS: items = items[:MAX_ITEMS]
print(f'[Batch] {len(items)} prompts queued. Variant: {BATCH_VARIANT}')

# Resolve resolution
h, w = (BATCH_HEIGHT, BATCH_WIDTH)
for name, ph, pw in RES_PRESETS:
    if name == BATCH_RES_NAME and ph != 0:
        h, w = ph, pw
        break
print(f'[Batch] Resolution: {h}x{w}. Output: {DRIVE_OUT}')

ENGINE.load(variant=BATCH_VARIANT, offload=BATCH_OFFLOAD)
ok = 0; fail = 0
for i, prompt in enumerate(items, 1):
    try:
        t0 = time.time()
        out_path = ENGINE.generate(
            prompt=prompt,
            image=None,
            height=h,
            width=w,
            num_frames=int(round(BATCH_DURATION * 24)),
            num_inference_steps=BATCH_STEPS,
            seed=BATCH_SEED,
        )
        safe = ''.join(c if c.isalnum() else '_' for c in prompt[:30]).strip('_') or f'item_{i:02d}'
        final = DRIVE_OUT / f'{i:03d}_{safe}.mp4'
        shutil.copy2(out_path, str(final))
        print(f'  [{i:03d}] {prompt[:60]}... -> {final.name} ({time.time() - t0:.1f}s)')
        ok += 1
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        print(f'  [{i:03d}] EXCEPTION: {e}'); traceback.print_exc(); fail += 1

print(f'\n[Batch] DONE: {ok} OK / {fail} failed / {len(items)} total -> {DRIVE_OUT}')